# 🛒 Analyse Exploratoire des Habitudes d'Achat — Données Instacart

## Objectif du projet

Ce projet analyse les données de la plateforme **Instacart** (3,4 millions de commandes, 200 000 utilisateurs) afin de :
- Comprendre les **habitudes d'achat** des clients (quand, quoi, combien)
- Identifier les **produits les plus populaires** et les tendances bio
- Fournir des **recommandations stratégiques** pour optimiser les ventes

> **Source des données** : [Instacart Market Basket Analysis — Kaggle](https://www.kaggle.com/c/instacart-market-basket-analysis)


## Sommaire

1. [Import et configuration](#1-import-et-configuration)
2. [Exploration des données](#2-exploration-des-données)
3. [Nettoyage des données](#3-nettoyage-des-données)
4. [Analyse 1 — Quand est-ce que les utilisateurs achètent ?](#4-analyse-1--quand-achètent-les-utilisateurs-)
5. [Analyse 2 — Combien d'articles par commande ?](#5-analyse-2--combien-darticles-par-commande-)
6. [Analyse 3 — Quels sont les produits les plus vendus ?](#6-analyse-3--produits-les-plus-vendus)
7. [Analyse 4 — Fréquence de commande des utilisateurs](#7-analyse-4--fréquence-de-commande)
8. [Analyse 5 — Premier produit ajouté au panier](#8-analyse-5--premier-produit-ajouté-au-panier)
9. [Analyse 6 — Ratio de produits bio dans les commandes](#9-analyse-6--ratio-de-produits-bio)
10. [Analyse 7 — Produits distincts par département](#10-analyse-7--produits-distincts-par-département)
11. [Analyse croisée — Heatmap heure × jour](#11-analyse-croisée--heatmap-heure--jour)
12. [Conclusion et recommandations](#12-conclusion-et-recommandations)


## 1. Import et configuration

Nous utilisons **Pandas** pour la manipulation de données, **Seaborn** et **Matplotlib** pour les visualisations statiques, et **Plotly** pour les graphiques interactifs.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

# Configuration globale
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Couleurs personnalisées
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'accent': '#F18F01',
    'success': '#2ECC71',
    'dark': '#2C3E50'
}

print('✅ Bibliothèques chargées avec succès')


## 2. Exploration des données

### Description des datasets

| Dataset | Description | Colonnes clés |
|---------|-------------|---------------|
| `aisles.csv` | Référentiel des rayons | `aisle_id`, `aisle` |
| `departments.csv` | Référentiel des départements | `department_id`, `department` |
| `orders.csv` | Métadonnées des commandes | `order_id`, `user_id`, `order_dow`, `order_hour_of_day` |
| `order_products.csv` | Produits par commande | `order_id`, `product_id`, `add_to_cart_order`, `reordered` |
| `products.csv` | Catalogue de produits | `product_id`, `product_name`, `aisle_id`, `department_id` |


### Chargement des données


In [ ]:
# Chargement des 5 datasets
aisles = pd.read_csv('DATASET/aisles.csv')
departments = pd.read_csv('DATASET/departments.csv')
orders = pd.read_csv('DATASET/orders.csv')
order_products = pd.read_csv('DATASET/order_products.csv')
products = pd.read_csv('DATASET/products.csv')

print(f'Aisles        : {aisles.shape[0]:>10,} lignes × {aisles.shape[1]} colonnes')
print(f'Departments   : {departments.shape[0]:>10,} lignes × {departments.shape[1]} colonnes')
print(f'Orders        : {orders.shape[0]:>10,} lignes × {orders.shape[1]} colonnes')
print(f'Order_products: {order_products.shape[0]:>10,} lignes × {order_products.shape[1]} colonnes')
print(f'Products      : {products.shape[0]:>10,} lignes × {products.shape[1]} colonnes')


### Aperçu de chaque dataset


In [ ]:
# --- AISLES (Rayons) ---
print('=== AISLES ===')
print(f'Dimensions : {aisles.shape}')
print(f'Valeurs manquantes : {aisles.isnull().sum().sum()}')
print(f'Doublons : {aisles.duplicated().sum()}')
display(aisles.head())


In [ ]:
# --- DEPARTMENTS (Départements) ---
print('=== DEPARTMENTS ===')
print(f'Dimensions : {departments.shape}')
print(f'Valeurs manquantes : {departments.isnull().sum().sum()}')
print(f'Doublons : {departments.duplicated().sum()}')
display(departments.head())


In [ ]:
# --- ORDERS (Commandes) ---
print('=== ORDERS ===')
print(f'Dimensions : {orders.shape}')
print(f'\nValeurs manquantes par colonne :')
print(orders.isnull().sum())
print(f'\nDoublons : {orders.duplicated().sum()}')
display(orders.head())
display(orders.describe())


In [ ]:
# --- ORDER_PRODUCTS (Produits commandés) ---
print('=== ORDER_PRODUCTS ===')
print(f'Dimensions : {order_products.shape}')
print(f'Valeurs manquantes : {order_products.isnull().sum().sum()}')
print(f'Doublons : {order_products.duplicated().sum()}')
display(order_products.head())


In [ ]:
# --- PRODUCTS (Produits) ---
print('=== PRODUCTS ===')
print(f'Dimensions : {products.shape}')
print(f'Valeurs manquantes : {products.isnull().sum().sum()}')
print(f'Doublons : {products.duplicated().sum()}')
display(products.head())


## 3. Nettoyage des données

Avant de passer à l'analyse, nous devons vérifier et traiter :
- Les **valeurs manquantes** (notamment `days_since_prior_order` dans `orders`)
- Les **doublons** éventuels
- La **cohérence des types** de données


In [ ]:
# Valeurs manquantes dans 'orders'
missing = orders.isnull().sum()
missing_pct = (orders.isnull().sum() / len(orders) * 100).round(2)
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage (%)': missing_pct})
print('Valeurs manquantes dans orders :')
display(missing_df[missing_df['Manquantes'] > 0])


In [ ]:
# La colonne 'days_since_prior_order' est NaN pour la première commande de chaque utilisateur.
# C'est un comportement attendu : la première commande n'a pas de commande précédente.

first_orders = orders[orders['days_since_prior_order'].isnull()]
print(f"Nombre de premières commandes (NaN dans days_since_prior_order) : {len(first_orders):,}")
print(f"Nombre d'utilisateurs uniques : {orders['user_id'].nunique():,}")
print(f"\n→ Cohérent : chaque utilisateur a exactement une première commande sans historique.")


In [ ]:
# Vérification des doublons sur tous les datasets
datasets = {'aisles': aisles, 'departments': departments, 'orders': orders, 
            'order_products': order_products, 'products': products}

for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    print(f'{name:>16} : {dup_count} doublons')

print('\n✅ Aucun doublon détecté — les données sont propres.')


In [ ]:
# Vérification des types de données
print('Types de données dans orders :')
print(orders.dtypes)
print(f"\nColonne 'eval_set' — valeurs uniques : {orders['eval_set'].unique()}")
print("→ Cette colonne indique la partition train/test. Nous la conservons pour information.")


## 4. Analyse 1 — Quand achètent les utilisateurs ?

Nous analysons les commandes selon deux axes temporels :
- Le **jour de la semaine** (`order_dow` : 0 = Dimanche dans le dataset Instacart)
- L'**heure de la journée** (`order_hour_of_day` : 0–23)


In [ ]:
# Mapping des jours (Instacart : 0 = Dimanche)
day_mapping = {0: 'Dimanche', 1: 'Lundi', 2: 'Mardi', 3: 'Mercredi',
               4: 'Jeudi', 5: 'Vendredi', 6: 'Samedi'}

orders['day_name'] = orders['order_dow'].map(day_mapping)

# Nombre de commandes par jour
orders_by_day = orders['day_name'].value_counts().reindex(
    ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(orders_by_day.index, orders_by_day.values, 
              color=[COLORS['primary'] if v < orders_by_day.max() else COLORS['accent'] 
                     for v in orders_by_day.values],
              edgecolor='white', linewidth=0.5)

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, orders_by_day.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000, 
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_xlabel('Jour de la semaine', fontsize=13)
ax.set_ylabel('Nombre de commandes', fontsize=13)
ax.set_title('📅 Distribution des commandes par jour de la semaine', fontsize=15, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Distribution des commandes par heure de la journée
orders_by_hour = orders['order_hour_of_day'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(orders_by_hour.index, orders_by_hour.values, alpha=0.3, color=COLORS['primary'])
ax.plot(orders_by_hour.index, orders_by_hour.values, 
        marker='o', color=COLORS['primary'], linewidth=2, markersize=6)

# Annoter le pic
peak_hour = orders_by_hour.idxmax()
peak_val = orders_by_hour.max()
ax.annotate(f'Pic : {peak_hour}h\n({peak_val:,} commandes)', 
            xy=(peak_hour, peak_val), xytext=(peak_hour + 3, peak_val + 5000),
            arrowprops=dict(arrowstyle='->', color=COLORS['accent'], lw=2),
            fontsize=11, fontweight='bold', color=COLORS['accent'])

ax.set_xlabel('Heure de la journée', fontsize=13)
ax.set_ylabel('Nombre de commandes', fontsize=13)
ax.set_title('🕐 Distribution des commandes par heure', fontsize=15, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h}h' for h in range(24)])
sns.despine()
plt.tight_layout()
plt.show()


### 📌 Interprétation — Timing des achats

**Constats :**
- Le **dimanche** et le **lundi** sont les jours les plus actifs, suggérant un comportement de planification hebdomadaire.
- Les commandes sont concentrées entre **10h et 15h**, avec un pic net à **10h du matin**.
- L'activité chute drastiquement après **18h** et est quasi inexistante entre 1h et 5h.

**Recommandations :**
- 🎯 **Promotions flash** le samedi soir pour stimuler les commandes du dimanche/lundi
- 📧 **Notifications push** entre 9h et 10h pour capter le pic de trafic
- 🌙 Les heures creuses (20h–6h) pourraient être exploitées avec des **offres nocturnes** exclusives


## 5. Analyse 2 — Combien d'articles par commande ?

Nous analysons la taille des paniers pour comprendre le comportement d'achat.


In [ ]:
# Nombre d'articles par commande
items_per_order = order_products.groupby('order_id')['product_id'].count()

print('📊 Statistiques descriptives du nombre d\'articles par commande :')
print(f'   Moyenne  : {items_per_order.mean():.1f} articles')
print(f'   Médiane  : {items_per_order.median():.0f} articles')
print(f'   Minimum  : {items_per_order.min()} article')
print(f'   Maximum  : {items_per_order.max()} articles')
print(f'   Écart-type : {items_per_order.std():.1f}')


In [ ]:
# Distribution de la taille des paniers
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(items_per_order, bins=50, color=COLORS['primary'], edgecolor='white',
        alpha=0.85, linewidth=0.5)
ax.axvline(items_per_order.mean(), color=COLORS['accent'], linestyle='--', linewidth=2,
           label=f'Moyenne : {items_per_order.mean():.1f} articles')
ax.axvline(items_per_order.median(), color=COLORS['secondary'], linestyle='--', linewidth=2,
           label=f'Médiane : {items_per_order.median():.0f} articles')

ax.set_xlabel('Nombre d\'articles par commande', fontsize=13)
ax.set_ylabel('Fréquence', fontsize=13)
ax.set_title('🛒 Distribution du nombre d\'articles par commande', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Catégorisation des commandes par taille (basée sur les quartiles)
q25 = items_per_order.quantile(0.25)
q75 = items_per_order.quantile(0.75)

def categorize_order(count):
    """Classifie une commande en petite, moyenne ou grande."""
    if count <= q25:
        return f'Petite (1–{int(q25)})'
    elif count <= q75:
        return f'Moyenne ({int(q25)+1}–{int(q75)})'
    else:
        return f'Grande ({int(q75)+1}+)'

order_categories = items_per_order.apply(categorize_order)
cat_counts = order_categories.value_counts(normalize=True) * 100

print(f'Seuils de catégorisation (quartiles) :')
print(f'   Petite commande  : 1 à {int(q25)} articles')
print(f'   Commande moyenne : {int(q25)+1} à {int(q75)} articles')
print(f'   Grande commande  : {int(q75)+1}+ articles')
print(f'\nRépartition :')
for cat, pct in cat_counts.items():
    print(f'   {cat} : {pct:.1f}%')


In [ ]:
# Visualisation en pie chart
fig, ax = plt.subplots(figsize=(8, 8))
colors_pie = [COLORS['success'], COLORS['primary'], COLORS['accent']]
wedges, texts, autotexts = ax.pie(
    cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
    colors=colors_pie, startangle=90, textprops={'fontsize': 11}
)
plt.setp(autotexts, fontweight='bold', color='white', fontsize=12)
ax.set_title('📦 Répartition des commandes par taille', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### 📌 Interprétation — Taille des paniers

**Constats :**
- En moyenne, une commande contient **~10 articles** (médiane similaire).
- La distribution est **asymétrique à droite** : quelques commandes géantes (jusqu'à 145 articles) tirent la moyenne vers le haut.
- Les commandes de taille moyenne représentent la **majorité** du volume.

**Recommandations :**
- 📦 **Offres « panier complet »** : proposer des bundles de 10–15 produits pour maximiser la taille moyenne des paniers
- 🎁 **Seuil de livraison gratuite** : positionner le seuil juste au-dessus de la médiane pour inciter à l'ajout d'articles
- 📊 Les très petites commandes (1–5 articles) révèlent peut-être des achats impulsifs — opportunité pour du **cross-selling**


## 6. Analyse 3 — Produits les plus vendus

Quels sont les produits préférés des clients ?


In [ ]:
# Top 20 des produits les plus vendus
product_sales = order_products.merge(products, on='product_id', how='left')
top_products = product_sales['product_name'].value_counts().head(20).reset_index()
top_products.columns = ['product_name', 'total_orders']

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_products, x='total_orders', y='product_name',
            palette='viridis', ax=ax, edgecolor='white')

# Ajouter les valeurs
for i, (val, name) in enumerate(zip(top_products['total_orders'], top_products['product_name'])):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Nombre de commandes', fontsize=13)
ax.set_ylabel('')
ax.set_title('🏆 Top 20 des produits les plus vendus', fontsize=15, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Taux de réachat (reorder rate) par produit — Top 20
reorder_rate = product_sales.groupby('product_name')['reordered'].mean()
top_reorder = reorder_rate[product_sales['product_name'].value_counts().head(20).index]
top_reorder = top_reorder.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors_reorder = [COLORS['success'] if v > 0.5 else COLORS['primary'] for v in top_reorder.values]
ax.barh(top_reorder.index, top_reorder.values * 100, color=colors_reorder, edgecolor='white')

for i, val in enumerate(top_reorder.values):
    ax.text(val * 100 + 0.5, i, f'{val*100:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Taux de réachat (%)', fontsize=13)
ax.set_ylabel('')
ax.set_title('🔄 Taux de réachat des 20 produits les plus populaires', fontsize=15, fontweight='bold')
ax.axvline(50, color='gray', linestyle=':', alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()


### 📌 Interprétation — Produits populaires

**Constats :**
- Les **fruits et légumes frais** (bananes, fraises, épinards bio) dominent largement le Top 20.
- Les produits **biologiques** représentent une part significative des best-sellers.
- Le taux de réachat est très élevé (>60%) pour ces produits, confirmant un achat récurrent.

**Recommandations :**
- 🥦 **Mettre en avant les produits frais** en page d'accueil — ce sont les moteurs de trafic
- 🔄 Les produits à fort taux de réachat sont idéaux pour des **abonnements automatiques**
- 📉 Les produits populaires mais avec un faible taux de réachat méritent une investigation (qualité ? saisonnalité ?)


## 7. Analyse 4 — Fréquence de commande

À quelle fréquence les utilisateurs passent-ils commande ? Quel est le délai moyen entre deux commandes ?


In [ ]:
# Exclure les premières commandes (sans historique)
orders_with_history = orders.dropna(subset=['days_since_prior_order']).copy()

# Délai moyen entre commandes par utilisateur
user_frequency = (orders_with_history
                  .groupby('user_id')['days_since_prior_order']
                  .mean()
                  .reset_index(name='avg_days_between_orders'))

print(f'📊 Statistiques sur la fréquence de commande :')
print(f'   Délai moyen global : {user_frequency["avg_days_between_orders"].mean():.1f} jours')
print(f'   Médiane            : {user_frequency["avg_days_between_orders"].median():.1f} jours')
print(f'   Utilisateur le plus fidèle : commande tous les {user_frequency["avg_days_between_orders"].min():.1f} jours')


In [ ]:
# Distribution de la fréquence de commande
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(user_frequency['avg_days_between_orders'], bins=30,
        color=COLORS['primary'], edgecolor='white', alpha=0.85)
ax.axvline(user_frequency['avg_days_between_orders'].mean(), 
           color=COLORS['accent'], linestyle='--', linewidth=2,
           label=f'Moyenne : {user_frequency["avg_days_between_orders"].mean():.1f} jours')

ax.set_xlabel('Jours moyens entre deux commandes', fontsize=13)
ax.set_ylabel('Nombre d\'utilisateurs', fontsize=13)
ax.set_title('⏱️ Fréquence moyenne de commande par utilisateur', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 des utilisateurs les plus fidèles (délai le plus court)
most_frequent = user_frequency.nsmallest(10, 'avg_days_between_orders')
print('🏅 Top 10 des utilisateurs les plus fréquents :')
display(most_frequent)


### 📌 Interprétation — Fréquence de commande

**Constats :**
- Les utilisateurs commandent en moyenne **tous les ~15 jours**, soit environ 2 fois par mois.
- On observe des pics à **7 jours** (hebdomadaire) et **30 jours** (mensuel), révélant deux segments distincts.
- Certains utilisateurs très fidèles commandent pratiquement tous les jours.

**Recommandations :**
- 📅 **Système de rappel** automatique basé sur le cycle habituel de chaque client
- 💎 **Programme de fidélité** ciblant les clients à fréquence hebdomadaire pour les récompenser
- 📈 Les clients mensuels représentent une opportunité de **montée en fréquence** (coupons valables 2 semaines)


## 8. Analyse 5 — Premier produit ajouté au panier

Quels produits les clients ajoutent-ils **en premier** dans leur panier ? Ces produits révèlent les priorités d'achat.


In [ ]:
# Produits ajoutés en premier au panier (add_to_cart_order == 1)
first_in_cart = order_products[order_products['add_to_cart_order'] == 1]
first_in_cart = first_in_cart.merge(products, on='product_id', how='left')

# Parmi ceux-ci, lesquels sont aussi des réachats ?
first_reordered = first_in_cart[first_in_cart['reordered'] == 1]

top_first = (first_reordered
             .groupby('product_name')['order_id']
             .count()
             .reset_index(name='count')
             .sort_values('count', ascending=False)
             .head(20))

print(f'Total de produits ajoutés en premier : {len(first_in_cart):,}')
print(f'Dont réachats : {len(first_reordered):,} ({len(first_reordered)/len(first_in_cart)*100:.1f}%)')


In [ ]:
# Visualisation des top 20 premiers produits ajoutés au panier (réachats)
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_first, x='count', y='product_name',
            palette='magma_r', ax=ax, edgecolor='white')

for i, val in enumerate(top_first['count']):
    ax.text(val + 200, i, f'{val:,}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Nombre de fois ajouté en premier', fontsize=13)
ax.set_ylabel('')
ax.set_title('🥇 Top 20 — Produits ajoutés en premier au panier (réachats)', fontsize=15, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


### 📌 Interprétation — Premiers produits au panier

**Constats :**
- Les produits ajoutés en premier sont majoritairement des **produits du quotidien** (bananes, lait, pain).
- Ces mêmes produits sont aussi les plus **réachetés**, confirmant une habitude routinière.
- Les produits bio figurent en bonne place, montrant que les clients bio sont très fidèles.

**Recommandations :**
- 🛒 Proposer une fonctionnalité **"Ma liste habituelle"** pré-remplie avec ces produits
- 📱 Ces produits devraient être en **accès rapide** sur l'interface utilisateur
- 🔁 Potentiel élevé pour des **abonnements récurrents** sur ces références


## 9. Analyse 6 — Ratio de produits bio

Quelle place occupent les produits biologiques dans le catalogue et dans les commandes ?


In [ ]:
# Identifier les produits bio (contiennent 'Organic' dans le nom)
products['is_organic'] = products['product_name'].str.contains('Organic', case=False, na=False)

organic_count = products['is_organic'].value_counts()
organic_pct = products['is_organic'].value_counts(normalize=True) * 100

print('🌱 Produits bio dans le catalogue :')
print(f'   Bio       : {organic_count[True]:,} produits ({organic_pct[True]:.1f}%)')
print(f'   Non-bio   : {organic_count[False]:,} produits ({organic_pct[False]:.1f}%)')


In [ ]:
# Distribution dans le catalogue
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Graphique 1 : Catalogue
labels_cat = ['Non-Bio', 'Bio']
colors_cat = [COLORS['primary'], COLORS['success']]
axes[0].pie(organic_count.values, labels=labels_cat, autopct='%1.1f%%',
            colors=colors_cat, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('🏪 Part des produits bio\ndans le catalogue', fontsize=13, fontweight='bold')

# Graphique 2 : Dans les commandes
order_with_organic = order_products.merge(products[['product_id', 'is_organic']], on='product_id')
orders_organic_flag = order_with_organic.groupby('order_id')['is_organic'].any()
orders_organic_count = orders_organic_flag.value_counts()

labels_ord = ['Sans bio', 'Avec bio']
colors_ord = [COLORS['secondary'], COLORS['success']]
axes[1].pie(orders_organic_count.values, labels=labels_ord, autopct='%1.1f%%',
            colors=colors_ord, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('🛒 Part des commandes contenant\nau moins un produit bio', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n→ Bien que les produits bio ne représentent que {organic_pct[True]:.1f}% du catalogue,')
print(f'  ils apparaissent dans {orders_organic_count[True]/orders_organic_count.sum()*100:.1f}% des commandes.')


### 📌 Interprétation — Produits bio

**Constats :**
- Les produits bio ne représentent qu'une **fraction du catalogue**, mais ils sont présents dans une **part significative des commandes**.
- Les produits bio figurent parmi les **plus vendus** et les **plus réachetés** — disproportionnellement populaires.
- Cela révèle une **forte demande** non entièrement satisfaite par l'offre actuelle.

**Recommandations :**
- 🌿 **Élargir le catalogue bio** — la demande dépasse clairement l'offre
- 🏷️ Créer une **section "Bio" dédiée** avec mise en avant sur la page d'accueil
- 💰 Les clients bio montrant une forte fidélité, une **gamme premium bio** pourrait augmenter le panier moyen


## 10. Analyse 7 — Produits distincts par département

Comment se répartit la diversité de l'offre entre les différents départements ?


In [ ]:
# Nombre de produits uniques par département
products_with_dept = products.merge(departments, on='department_id', how='left')
dept_product_count = (products_with_dept
                      .groupby('department')['product_name']
                      .nunique()
                      .reset_index(name='unique_products')
                      .sort_values('unique_products', ascending=True))

fig, ax = plt.subplots(figsize=(12, 8))
colors_dept = sns.color_palette('viridis', n_colors=len(dept_product_count))
ax.barh(dept_product_count['department'], dept_product_count['unique_products'],
        color=colors_dept, edgecolor='white')

for i, val in enumerate(dept_product_count['unique_products']):
    ax.text(val + 50, i, f'{val:,}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Nombre de produits distincts', fontsize=13)
ax.set_ylabel('')
ax.set_title('🏬 Nombre de produits distincts par département', fontsize=15, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


### 📌 Interprétation — Diversité par département

**Constats :**
- Le département **Personal Care** offre la plus grande diversité de produits, bien qu'il ne figure pas parmi les départements les plus vendeurs.
- Les départements à forte rotation (fruits, légumes, produits laitiers) ont une diversité relativement modérée.

**Recommandations :**
- 📊 La **disproportionnalité** entre diversité d'offre et volume de ventes suggère un potentiel de rationalisation du catalogue Personal Care
- 🍎 Les départements à forte vente mais faible diversité pourraient bénéficier d'un **élargissement ciblé** de gamme


## 11. Analyse croisée — Heatmap heure × jour

Cette analyse croisée permet de visualiser les **patterns temporels fins** des achats.


In [ ]:
# Heatmap : nombre de commandes par heure et par jour
heatmap_data = orders.groupby(['order_dow', 'order_hour_of_day']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='order_dow', columns='order_hour_of_day', values='count')

# Renommer les index avec les noms des jours
heatmap_pivot.index = [day_mapping[i] for i in heatmap_pivot.index]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(heatmap_pivot, cmap='YlOrRd', annot=False, fmt='d',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Nombre de commandes'})

ax.set_xlabel('Heure de la journée', fontsize=13)
ax.set_ylabel('Jour de la semaine', fontsize=13)
ax.set_title('🗓️ Heatmap — Commandes par heure et jour de la semaine', fontsize=15, fontweight='bold')
ax.set_xticklabels([f'{h}h' for h in range(24)], rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Corrélation entre taille du panier et fréquence de commande
# Calculer la taille moyenne du panier par utilisateur
user_basket = (order_products
               .groupby('order_id')['product_id'].count()
               .reset_index(name='basket_size'))

user_basket = user_basket.merge(orders[['order_id', 'user_id']], on='order_id')
user_avg_basket = user_basket.groupby('user_id')['basket_size'].mean().reset_index(name='avg_basket')

# Joindre avec la fréquence de commande
user_analysis = user_frequency.merge(user_avg_basket, on='user_id')

# Scatter plot avec échantillonnage (trop de points sinon)
sample = user_analysis.sample(n=min(5000, len(user_analysis)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(sample['avg_days_between_orders'], sample['avg_basket'],
                     alpha=0.3, s=15, color=COLORS['primary'])

# Corrélation
corr = user_analysis['avg_days_between_orders'].corr(user_analysis['avg_basket'])

ax.set_xlabel('Jours moyens entre deux commandes', fontsize=13)
ax.set_ylabel('Taille moyenne du panier', fontsize=13)
ax.set_title(f'📈 Corrélation fréquence × taille du panier (r = {corr:.3f})', 
             fontsize=15, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

print(f'Coefficient de corrélation de Pearson : r = {corr:.3f}')
if abs(corr) < 0.1:
    print('→ Corrélation très faible : la fréquence de commande et la taille du panier sont quasiment indépendantes.')
elif corr > 0:
    print('→ Corrélation positive : les clients qui commandent moins souvent ont tendance à faire de plus gros paniers.')
else:
    print('→ Corrélation négative : les clients fréquents tendent à faire des paniers plus petits.')


### 📌 Interprétation — Analyses croisées

**Constats :**
- La **heatmap temporelle** confirme le double pic dimanche/lundi en fin de matinée.
- Les **soirées de semaine** sont les périodes les plus creuses — un levier sous-exploité.
- L'analyse de corrélation fréquence/taille du panier révèle si ces deux comportements sont liés.

**Recommandations :**
- 🎯 Cibler les créneaux **dimanche 10h–14h** pour des mises en avant produit
- 🌙 Tester des **promotions en soirée** (18h–21h) pour étendre la fenêtre d'achat


## 12. Conclusion et recommandations

### 🔑 Résumé des insights principaux

| # | Insight | Impact business |
|---|---------|----------------|
| 1 | Les achats se concentrent le **dimanche/lundi matin** | Optimiser les campagnes marketing sur ce créneau |
| 2 | Le panier moyen contient **~10 articles** | Proposer des bundles de 10-15 produits |
| 3 | Les **fruits/légumes frais** dominent les ventes | Investir dans la fraîcheur et la qualité de ces rayons |
| 4 | Les clients commandent tous les **~15 jours** | Mettre en place des rappels bimensuels personnalisés |
| 5 | Les **bananes** sont le produit le plus ajouté en premier | Exploiter ce « produit d'appel » pour des ventes croisées |
| 6 | Le **bio** surperforme malgré une offre limitée | Élargir le catalogue bio |
| 7 | **Personal Care** a la plus grande diversité de produits | Rationaliser ce département vs. ventes réelles |

### 🚀 Recommandations stratégiques

1. **Stratégie temporelle** : Concentrer le budget marketing sur le dimanche/lundi matin et tester des promotions en soirée.

2. **Personnalisation** : Implémenter un système de rappel intelligent basé sur le cycle d'achat individuel de chaque client.

3. **Croissance bio** : Les produits bio sont surreprésentés dans les commandes par rapport à leur part dans le catalogue — élargir l'offre bio et créer une vitrine dédiée répondrait à cette demande latente.

4. **Optimisation du panier** : Utiliser le cross-selling sur les produits ajoutés en premier pour augmenter la valeur du panier.

5. **Programme de fidélité** : Récompenser les clients hebdomadaires et inciter les mensuels à augmenter leur fréquence.

### 📊 Pistes d'approfondissement

- **Segmentation RFM** (Récence, Fréquence, Montant) pour identifier les segments clients à fort potentiel
- **Analyse de panier** (Market Basket Analysis) avec l'algorithme Apriori pour le cross-selling
- **Modèle prédictif** de réachat pour anticiper les besoins des clients
- **Analyse de cohortes** pour mesurer la rétention dans le temps

---

*Analyse réalisée par Kristen OKE — Data Analyst* 
 
*Technologies : Python, Pandas, Seaborn, Matplotlib, Plotly*
